# Week 6 · Exercises

Each exercise is a function; run the check cell beneath it. The set-up cell must run first.
Solutions: `solutions/02_solutions.ipynb`.

**Two rules for this week.** (1) Functions that run in a **child process** must come from [`workers.py`](workers.py), never from this notebook, so that your answers also work on Windows and macOS (tutorial, Section 4). Functions that run in **threads** may be defined here. Do not name a variable `workers`: it would hide the module. (2) The checks with a time limit only test that tasks overlap; they leave plenty of margin, so a slow machine still passes.

In [ ]:
import os, sys, time, math, inspect, threading, queue, multiprocessing
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor
from pathlib import Path
import pandas as pd

HERE = next(p / "week06-parallelism" for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "week06-parallelism" / "workers.py").exists())
sys.path.insert(0, str(HERE))          # so that "import workers" works here and in child processes
import workers

DATA = next(p / "data" / "generated" for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "make_data.py").exists())
assert (DATA / "students.csv").exists(), "run  python data/make_data.py  from the repository root first"
PATHS = workers.make_student_files(DATA, n_files=4)     # students1.csv ... students4.csv
print("start method:", multiprocessing.get_start_method(), "| cores:", multiprocessing.cpu_count(),
      "| files:", [p.name for p in PATHS])

### Exercise 1 · How many workers?

`pool_size(n_tasks)` returns how many worker processes to start for `n_tasks` tasks: never more than the number of
usable CPU cores (slide 15: too many processes only adds overhead), never more than the number of tasks, and at
least 1. Count the cores with `multiprocessing.cpu_count()` (the slides) or, on Python 3.13+, `os.process_cpu_count()`.

In [ ]:
def pool_size(n_tasks):
    ...

In [ ]:
cores = getattr(os, "process_cpu_count", multiprocessing.cpu_count)()
assert pool_size(1) == 1, "one task needs one worker"
assert pool_size(0) == 1, "always start at least one worker"
assert pool_size(1000) == cores, f"never more workers than the {cores} cores"
assert pool_size(cores + 3) == cores and pool_size(max(1, cores - 1)) == max(1, cores - 1)
print("✓ exercise 1 -", cores, "cores")

### Exercise 2 · Processes and a Queue

`squares_via_processes(numbers)` starts **one `multiprocessing.Process` per number**, each running
`workers.square_to_queue(n, q)`, which puts the pair `(n, n * n)` into a shared `multiprocessing.Queue` `q`. Collect
one item per process, join the processes, and return a dict `{n: n * n}`. Remember (tutorial, Section 5.2): take the
items out of the queue before joining.

In [ ]:
def squares_via_processes(numbers):
    ...

In [ ]:
r = squares_via_processes([2, 3, 7])
assert isinstance(r, dict), "return a dict {n: n * n}"
assert r == {2: 4, 3: 9, 7: 49}, f"expected {{2: 4, 3: 9, 7: 49}}, got {r}"
assert squares_via_processes([10]) == {10: 100}
print("✓ exercise 2")

### Exercise 3 · `apply_async()` instead of `apply()`

`totals_async(sizes)` uses a `multiprocessing.Pool` with **2** processes to run `workers.count_total_quiet(n)` for
every `n` in `sizes`, **asynchronously**, and returns the list of results in the order of `sizes`. Each call
`count_total_quiet(n)` takes about `n × 0.2` s, so for `[6, 6]` the synchronous `apply()` needs about 2.4 s and the
asynchronous version about 1.2 s. Close and join the pool (or use a `with` block and collect the results inside it).

In [ ]:
def totals_async(sizes):
    ...

In [ ]:
t0 = time.perf_counter()
r = totals_async([6, 6])
elapsed = time.perf_counter() - t0
assert r == [15, 15], f"count_total_quiet(6) is 0+1+2+3+4+5 = 15; got {r}"
assert elapsed < 2.2, f"took {elapsed:.1f} s: the two tasks ran one after the other (apply?), not at the same time (that takes at least 2.4 s)"
assert totals_async([3, 4, 5]) == [3, 6, 10], "return the results in the order of sizes"
print(f"✓ exercise 3 - {elapsed:.1f} s")

### Exercise 4 · `starmap()` with a context manager

`powers(bases, exponent)` returns `[b ** exponent for b in bases]`, computed in a `Pool` used as a **context
manager** (`with ... as pool:`), with `pool.starmap` and `workers.power(base, exponent)`. Each element passed to
`starmap` must be a tuple of arguments.

In [ ]:
def powers(bases, exponent):
    ...

In [ ]:
r = powers([1, 2, 3, 10], 3)
assert isinstance(r, list), "starmap returns a list"
assert r == [1, 8, 27, 1000], f"expected [1, 8, 27, 1000], got {r}"
assert powers([], 2) == [] and powers([5], 0) == [1]
print("✓ exercise 4")

### Exercise 5 · Errors in workers

`safe_reciprocals(xs)` computes `workers.reciprocal(x)` (that is, `1 / x`) for every `x` with
`pool.apply_async`, and returns the list of results, with `None` where the worker raised `ZeroDivisionError`. The
exception is re-raised by `.get()` (slide 16), so that is where your `try`/`except` goes.

In [ ]:
def safe_reciprocals(xs):
    ...

In [ ]:
r = safe_reciprocals([4, 0, 2, 0])
assert r == [0.25, None, 0.5, None], f"expected [0.25, None, 0.5, None], got {r}"
assert safe_reciprocals([1]) == [1.0]
print("✓ exercise 5")

### Exercise 6 · Parallel data loading (the syllabus task)

`load_parallel(paths)` reads every CSV file in `paths` **in a process pool** (use `pool.map` with
`workers.read_csv_file`) and returns one DataFrame with all the rows, in the order of `paths`, and a fresh index
`0, 1, 2, …`. `PATHS` (from the set-up cell) holds the four parts of `students.csv`.

In [ ]:
def load_parallel(paths):
    ...

In [ ]:
assert "Pool" in inspect.getsource(load_parallel), "read the files in a process pool (multiprocessing.Pool)"
r = load_parallel(PATHS)
expected = pd.concat([pd.read_csv(p) for p in PATHS], ignore_index=True)
assert isinstance(r, pd.DataFrame), "return a DataFrame"
assert r.shape == expected.shape, f"expected shape {expected.shape}, got {r.shape}"
pd.testing.assert_frame_equal(r, expected)          # same rows, same order, index 0..n-1
assert len(load_parallel(PATHS[:2])) == len(pd.read_csv(PATHS[0])) + len(pd.read_csv(PATHS[1]))
print("✓ exercise 6 -", r.shape)

### Exercise 7 · Threads for I/O-bound work

`download_all(urls)` "downloads" every URL with `workers.fake_download(url)` (it waits 0.3 s, then returns
`(url, size)`) using a `ThreadPoolExecutor`, and returns the list of results **in the order of `urls`**. Six URLs
one after the other take 1.8 s; with one thread per URL they take about 0.3 s, and with 2 threads about 0.9 s.
The check accepts any version in which the downloads overlap.

In [ ]:
def download_all(urls):
    ...

In [ ]:
urls = [f"https://example.org/page{i}" for i in range(6)]
t0 = time.perf_counter()
r = download_all(urls)
elapsed = time.perf_counter() - t0
assert isinstance(r, list) and len(r) == 6, "return one result per URL, as a list"
assert [u for u, _ in r] == urls, "keep the order of urls"
assert elapsed < 1.5, f"took {elapsed:.1f} s: the downloads did not overlap (one after the other takes 1.8 s)"
print(f"✓ exercise 7 - {elapsed:.2f} s")

### Exercise 8 · A race condition and a `Lock`

The function below runs `n_threads` threads that each make `n_deposits` deposits of 1 into a shared account.
Run it: the balance comes out far too low, because deposits overwrite each other (tutorial, Section 9.1). Fix it with
a `threading.Lock` so that the result is always exactly `n_threads * n_deposits`. Keep the `time.sleep`: it stands
for real work between reading and writing.

In [ ]:
def deposit_all(n_threads, n_deposits):
    account = {"balance": 0}

    def deposit():
        for _ in range(n_deposits):
            current = account["balance"]
            time.sleep(0.0001)                  # another thread may run here
            account["balance"] = current + 1

    threads = [threading.Thread(target=deposit) for _ in range(n_threads)]
    for t in threads:
        t.start()
    for t in threads:
        t.join()
    return account["balance"]

print(deposit_all(4, 50), "(should be 200)")

In [ ]:
results = [deposit_all(4, 50) for _ in range(3)]
assert results == [200, 200, 200], f"expected 200 every time, got {results}: protect the read-and-write with a Lock"
assert deposit_all(3, 30) == 90
print("✓ exercise 8")

### Exercise 9 · A `Queue` of files for worker threads

`count_rows(paths, n_workers)` starts `n_workers` threads. The main thread puts every path into a `queue.Queue`,
followed by one `None` per worker (the sentinel). Each worker repeatedly takes a path, stops at `None`, and otherwise
reads the file with `pd.read_csv` and records `(file name, number of rows)`. Return a dict
`{file name: number of rows}`, such as `{"students1.csv": 52, ...}`. Record the results in a second queue, or in a
dict protected by a `Lock`. If the cell never finishes, a worker is waiting for a sentinel: interrupt, restart the
kernel, and put one `None` per worker.

In [ ]:
def count_rows(paths, n_workers=2):
    ...

In [ ]:
before = threading.active_count()
r = count_rows(PATHS, n_workers=2)
expected = {p.name: len(pd.read_csv(p)) for p in PATHS}
assert isinstance(r, dict), "return a dict {file name: rows}"
assert r == expected, f"expected {expected}, got {r}"
assert count_rows(PATHS[:1], n_workers=3) == {PATHS[0].name: expected[PATHS[0].name]}, "more workers than files must also work"
assert threading.active_count() <= before, "every worker thread must stop (one sentinel per worker)"
print("✓ exercise 9", r)

### Exercise 10 · `ProcessPoolExecutor` for CPU-bound work

`factorial_sizes(numbers)` returns the number of decimal digits of `n!` for every `n` in `numbers`, in order,
computed with a `ProcessPoolExecutor` with 2 workers and `workers.factorial_digits`.

In [ ]:
def factorial_sizes(numbers):
    ...

In [ ]:
assert "ProcessPoolExecutor" in inspect.getsource(factorial_sizes), "use a ProcessPoolExecutor"
sys.set_int_max_str_digits(0)
nums = [10, 100, 1000, 5000]
r = factorial_sizes(nums)
assert isinstance(r, list), "return a list"
assert r == [len(str(math.factorial(n))) for n in nums], f"wrong digit counts: {r}"
assert r[:2] == [7, 158], "10! = 3628800 has 7 digits"
print("✓ exercise 10", r)